General Page Rank Algorithm

In [2]:
# imports
import numpy as np
from scipy.sparse import csr_array

In [10]:
# function to calculate ∑_k a[j][k]
def sum_for_M(A: csr_array, j: int) -> int:
    result = 0
    for k in range(A.shape[0]):
        result += A[j][k]
    return result

def construct_M(A: csr_array) -> csr_array:
    M = np.zeros((A.shape[0], A.shape[1]))
    
    for i in range(A.shape[0]):
        for j in range(A.shape[0]):
            local_sum = sum_for_M(A,j)
            # if local_sum is 0 then we don't want to divide by it
            if local_sum == 0:
                M[i][j] = 0
            else:
                M[i][j] = A[j][i]/sum_for_M(A,j)
    
    return csr_array(M)

def subtract_then_l1_norm(v1: np.array, v2: np.array) -> int:
    result_v = subtract_vectors(v1, v2)
    result_norm = 0
    for value in result_v:
        result_norm += abs(value)
    return result_norm

def subtract_vectors(v1: np.array, v2: np.array) -> np.array:
    result = np.zeros(v1.shape[0])
    for i in range(v1.shape[0]):
        result[i] = v2[i] - v1[i]
    return result

def edges_of_vertex_calculation(G: csr_array, vertex: int, beta: int, r_new: np.array):
    result = 0
    for i in range(G.shape[0]):
        if G[i][vertex] == 0:
            continue
        result += (beta * (r_new[i]/outdeg(G,i)))
    return result

def vector_sum(r_new_copy: np.array) -> int:
    result = 0
    for v in r_new_copy:
        result += v
    return result


def outdeg(G: csr_array, v: int) -> int:
    result = 0
    for u in G[v]:
        if u > 0:
            result += 1
    return result

def indeg(G: csr_array, v: int) -> int:
    result = 0
    for row in G:
        if row[v] > 0:
            result += 1
    return result


In [12]:
# lecture example graph as sparse matrix
A = csr_array(np.array([[0,1,0,0,0], [1,0,1,0,0], [0,0,0,1,1], [0,0,0,0,0], [0,1,0,0,0]]))
M = construct_M(A)


In [13]:
def general_pagerank(G: csr_array, epsilon: float, beta: float) -> np.array:
    n = G.shape[0]
    M = construct_M(G)
    r_new = np.zeros(n)
    for i in range(n):
        r_new[i] = 1/n

    r_old = r_new

    print(f"Graph G: \n{G}\nMatrix M: \n{M}\n\nBeta: {beta}")

    iter = 0
    # while((subtract_then_l1_norm(r_old, r_new)) < epsilon):
    while(iter < 20):
        print(f"=================================\n\nIteration: {iter}\n\n{r_new.transpose()}\n\n=================================")
        r_new_hat = (beta*M) @ r_new
        D = vector_sum(r_new_hat)
        print(f"D: {D}")
        for v in range(n):
            r_new[v] = (r_new_hat[v] + ((1-D)/n))
        iter += 1
    
    return r_new

print(general_pagerank(A, 10e-6, 0.85))

# r_new_hat = np.zeros(n)
        # for v in range(G.shape[0]):
        #     if indeg(G, v) == 0:
        #         r_new_hat[v] = 0
        #     else:
        #         r_old[v] = r_new[v]
        #         r_new_hat[v] = edges_of_vertex_calculation(G, v, beta, r_new)

Graph G: 
<Compressed Sparse Row sparse array of dtype 'int64'
	with 6 stored elements and shape (5, 5)>
  Coords	Values
  (0, 1)	1
  (1, 0)	1
  (1, 2)	1
  (2, 3)	1
  (2, 4)	1
  (4, 1)	1
Matrix M: 
<Compressed Sparse Row sparse array of dtype 'float64'
	with 6 stored elements and shape (5, 5)>
  Coords	Values
  (0, 1)	0.5
  (1, 0)	1.0
  (1, 4)	1.0
  (2, 1)	0.5
  (3, 2)	0.5
  (4, 2)	0.5

Beta: 0.85

Iteration: 0

[0.2 0.2 0.2 0.2 0.2]

D: 0.6799999999999999

Iteration: 1

[0.149 0.404 0.149 0.149 0.149]

D: 0.7233499999999999

Iteration: 2

[0.22703  0.30863  0.22703  0.118655 0.118655]

D: 0.74914325

Iteration: 3

[0.1813391 0.3440036 0.1813391 0.1466591 0.1466591]

D: 0.725339765

Iteration: 4

[0.20113358 0.33373052 0.20113358 0.13200116 0.13200116]

D: 0.7377990101749999

Iteration: 5

[0.19427567 0.33560473 0.19427567 0.13792197 0.13792197]

D: 0.7327663270384999

Iteration: 6

[0.19607874 0.33581473 0.19607874 0.13601389 0.13601389]

D: 0.7343881906435326

Iteration: 7

[0.195843